# CheckMite mvp1v11 - YOLOv8m Small Object Detection (640 Tiling)

**파이프라인:**
1. 환경 설정 (GPU, Drive, ultralytics)
2. 설정값 정의
3. 병합 완료된 YOLO zip 데이터셋 로드
4. YOLOv8 폴더 구조로 변환
5. 이미지-라벨 정합성 정리
6. Train/Test 9:1 재분할
7. 640px 타일 데이터셋 생성 (negative 5% 유지)
8. YOLOv8m 학습 (early stopping)
9. 결과 시각화
10. 결과 저장

**학습 데이터셋:**
- `yolo_export_task30_20260505.zip`
- `yolo_export_task37_20260521.zip`
- `yolo_export_task38_20260505.zip`
- `yolo_export_task39_20260521.zip`


In [1]:
# === Cell 1: 환경 설정 ===
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics -q

import ultralytics
ultralytics.checks()


Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
Setup complete ✅ (12 CPUs, 167.1 GB RAM, 47.3/235.7 GB disk)


In [ ]:
# === Cell 1-1: 런타임 확인 ===
import os

print("현재 작업 경로:", os.getcwd())
!nvidia-smi


현재 작업 경로: /content
Fri May 22 11:26:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   35C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+----------------------------

In [2]:
# === Cell 2: 경로 및 설정 ===
import os

DATASET_ZIP_DIR = '/content/drive/MyDrive/checkmite/dataset'

DATASET_ZIPS = [
    'yolo_export_task30_20260505.zip',
    'yolo_export_task37_20260521.zip',
    'yolo_export_task38_20260505.zip',
    'yolo_export_task39_20260521.zip',
]

DATASET_ZIP_PATHS = [
    os.path.join(DATASET_ZIP_DIR, name)
    for name in DATASET_ZIPS
]

SAVE_DIR = "/content/drive/Shareddrives/checkmite/result/mvp1v11_output"
WORK_DIR = "/content/checkmite_mvp1v11"
RUN_PROJECT = "/content/drive/MyDrive/checkmite/runs"
RUN_NAME = "mvp1v11"

TILE_SIZE = 640
OVERLAP_RATIO = 0.25
MIN_BBOX_AREA_RATIO = 0.3
MIN_BBOX_PX = 4
NEGATIVE_TILE_RATIO = 0.05

TRAIN_RATIO = 0.9

CLASS_NAMES = {
    0: "predator",
    1: "prey"
}

MODEL = "yolov8m.pt"
IMGSZ = 640
BATCH = 16
EPOCHS = 300
PATIENCE = 50

print("Dataset zip files:")
for p in DATASET_ZIP_PATHS:
    print(f"  {p} | exists={os.path.exists(p)}")

Dataset zip files:
  /content/drive/MyDrive/checkmite/dataset/yolo_export_task30_20260505.zip | exists=True
  /content/drive/MyDrive/checkmite/dataset/yolo_export_task37_20260521.zip | exists=True
  /content/drive/MyDrive/checkmite/dataset/yolo_export_task38_20260505.zip | exists=True
  /content/drive/MyDrive/checkmite/dataset/yolo_export_task39_20260521.zip | exists=True


In [ ]:
# === Cell 2-1: Drive 내 checkmite 폴더 확인 ===
!ls -lah '/content/drive/MyDrive/checkmite'


In [3]:
# === Cell 3: Drive에 직접 병합 ===
import os
import zipfile
import shutil
from pathlib import Path
from collections import Counter

IMG_EXTS = ('.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG')

DRIVE_MERGED_DIR = "/content/drive/MyDrive/checkmite/dataset/merged_dataset"

MERGED_IMAGES_DIR = os.path.join(DRIVE_MERGED_DIR, "images", "All")
MERGED_LABELS_DIR = os.path.join(DRIVE_MERGED_DIR, "labels", "All")

EXTRACT_ROOT = "/content/extracted_zips"

# 기존 제거
if os.path.exists(DRIVE_MERGED_DIR):
    shutil.rmtree(DRIVE_MERGED_DIR)

if os.path.exists(EXTRACT_ROOT):
    shutil.rmtree(EXTRACT_ROOT)

os.makedirs(MERGED_IMAGES_DIR, exist_ok=True)
os.makedirs(MERGED_LABELS_DIR, exist_ok=True)
os.makedirs(EXTRACT_ROOT, exist_ok=True)

def find_dataset_root(extract_dir):
    for root, dirs, files in os.walk(extract_dir):
        if "images" in dirs and "labels" in dirs:
            return root
    raise RuntimeError(f"dataset root 못찾음: {extract_dir}")

merged_image_count = 0
merged_label_count = 0
missing_label_count = 0
bbox_counter = Counter()

for zip_path in DATASET_ZIP_PATHS:

    task_name = Path(zip_path).stem
    extract_dir = os.path.join(EXTRACT_ROOT, task_name)

    print(f"\n===== {task_name} =====")

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_dir)

    dataset_root = find_dataset_root(extract_dir)

    images_root = os.path.join(dataset_root, "images")
    labels_root = os.path.join(dataset_root, "labels")

    for split in os.listdir(images_root):

        img_dir = os.path.join(images_root, split)
        lbl_dir = os.path.join(labels_root, split)

        if not os.path.isdir(img_dir):
            continue

        print(f"split={split}")

        img_files = sorted([
            f for f in os.listdir(img_dir)
            if f.endswith(IMG_EXTS)
        ])

        for img_file in img_files:

            stem, ext = os.path.splitext(img_file)

            src_img = os.path.join(img_dir, img_file)
            src_lbl = os.path.join(lbl_dir, f"{stem}.txt")

            new_stem = f"{task_name}__{split}__{stem}"

            dst_img = os.path.join(
                MERGED_IMAGES_DIR,
                f"{new_stem}{ext.lower()}"
            )

            dst_lbl = os.path.join(
                MERGED_LABELS_DIR,
                f"{new_stem}.txt"
            )

            shutil.copy2(src_img, dst_img)
            merged_image_count += 1

            if not os.path.exists(src_lbl):
                missing_label_count += 1
                continue

            valid_lines = []

            with open(src_lbl, "r") as f:
                for line in f:

                    parts = line.strip().split()

                    if len(parts) < 5:
                        continue

                    try:
                        cls = int(float(parts[0]))
                        vals = [float(x) for x in parts[1:5]]
                    except:
                        continue

                    if cls < 0 or cls >= len(CLASS_NAMES):
                        continue

                    if not all(0 <= v <= 1 for v in vals):
                        continue

                    valid_lines.append(
                        f"{cls} "
                        f"{vals[0]:.6f} "
                        f"{vals[1]:.6f} "
                        f"{vals[2]:.6f} "
                        f"{vals[3]:.6f}"
                    )

                    bbox_counter[cls] += 1

            with open(dst_lbl, "w") as f:
                f.write("\n".join(valid_lines))

            merged_label_count += 1

    # zip 하나 끝나면 extracted 제거
    shutil.rmtree(extract_dir)

print("\n===== 병합 완료 =====")
print("images :", merged_image_count)
print("labels :", merged_label_count)
print("missing:", missing_label_count)
print("bbox   :", dict(bbox_counter))

print("\n저장 위치:")
print(DRIVE_MERGED_DIR)


===== yolo_export_task30_20260505 =====
split=Train

===== yolo_export_task37_20260521 =====
split=Train

===== yolo_export_task38_20260505 =====
split=Train

===== yolo_export_task39_20260521 =====
split=Test

===== 병합 완료 =====
images : 10015
labels : 9263
missing: 752
bbox   : {1: 6214, 0: 25217}

저장 위치:
/content/drive/MyDrive/checkmite/dataset/merged_dataset


In [5]:
# === Cell 4: YOLOv8 폴더 구조로 Train/Val 변환 ===
import os, shutil, random

dataset_dir = "/content/drive/MyDrive/checkmite/dataset/merged_dataset"

src_img = os.path.join(dataset_dir, "images", "All")
src_lbl = os.path.join(dataset_dir, "labels", "All")

if not os.path.exists(src_img):
    raise FileNotFoundError(f"이미지 폴더를 찾을 수 없습니다: {src_img}")
if not os.path.exists(src_lbl):
    raise FileNotFoundError(f"라벨 폴더를 찾을 수 없습니다: {src_lbl}")

for sub in ["images", "labels"]:
    target = os.path.join(WORK_DIR, sub)
    if os.path.exists(target):
        shutil.rmtree(target)

all_images = [f for f in os.listdir(src_img) if f.endswith(IMG_EXTS)]

valid_images = []
for img in all_images:
    stem = os.path.splitext(img)[0]
    label_path = os.path.join(src_lbl, f"{stem}.txt")
    if os.path.exists(label_path):
        valid_images.append(img)

random.seed(42)
random.shuffle(valid_images)

if not valid_images:
    raise RuntimeError("라벨과 매칭되는 이미지가 없습니다.")

split_idx = int(len(valid_images) * TRAIN_RATIO)
train_imgs = valid_images[:split_idx]
val_imgs = valid_images[split_idx:]

for split in ["train", "val"]:
    os.makedirs(os.path.join(WORK_DIR, "images", split), exist_ok=True)
    os.makedirs(os.path.join(WORK_DIR, "labels", split), exist_ok=True)

copied_labels = {"train": 0, "val": 0}

for split, imgs in [("train", train_imgs), ("val", val_imgs)]:
    for img in imgs:
        stem = os.path.splitext(img)[0]

        shutil.copy2(
            os.path.join(src_img, img),
            os.path.join(WORK_DIR, "images", split, img)
        )

        shutil.copy2(
            os.path.join(src_lbl, f"{stem}.txt"),
            os.path.join(WORK_DIR, "labels", split, f"{stem}.txt")
        )

        copied_labels[split] += 1

print(f"train: {len(train_imgs)}장, labels: {copied_labels['train']}개")
print(f"val:   {len(val_imgs)}장, labels: {copied_labels['val']}개")

train: 8336장, labels: 8336개
val:   927장, labels: 927개


In [6]:
# === Cell 5: 이미지-라벨 정합성 정리 ===
import glob

for cache in glob.glob(f"{WORK_DIR}/labels/**/*.cache", recursive=True):
    os.remove(cache)
    print(f"캐시 삭제: {cache}")

img_base = os.path.join(WORK_DIR, "images")
label_base = os.path.join(WORK_DIR, "labels")

splits = ["train", "val"]

for split in splits:
    img_path = os.path.join(img_base, split)
    label_path = os.path.join(label_base, split)

    imgs = set(os.path.splitext(f)[0] for f in os.listdir(img_path) if f.endswith(IMG_EXTS))
    labels = set(os.path.splitext(f)[0] for f in os.listdir(label_path) if f.endswith(".txt"))

    for name in imgs - labels:
        for ext in IMG_EXTS:
            p = os.path.join(img_path, f"{name}{ext}")
            if os.path.exists(p):
                os.remove(p)

    for name in labels - imgs:
        p = os.path.join(label_path, f"{name}.txt")
        if os.path.exists(p):
            os.remove(p)

    n_img = len([f for f in os.listdir(img_path) if f.endswith(IMG_EXTS)])
    n_label = len([f for f in os.listdir(label_path) if f.endswith(".txt")])
    print(f"{split:5s} | 이미지: {n_img}장 | 라벨: {n_label}개 | 일치: {n_img == n_label}")

    if n_img == 0:
        raise RuntimeError(f"{split} 데이터가 0장입니다.")

train | 이미지: 8336장 | 라벨: 8336개 | 일치: True
val   | 이미지: 927장 | 라벨: 927개 | 일치: True


In [7]:
# === Cell 6: 640px 타일 데이터셋 생성 ===
import cv2
import numpy as np
import yaml
import random
import shutil

def parse_yolo_label(label_path, img_w, img_h):
    boxes = []
    if not os.path.exists(label_path):
        return boxes

    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue

            cls = int(float(parts[0]))
            cx, cy, w, h = map(float, parts[1:5])

            x1 = (cx - w / 2) * img_w
            y1 = (cy - h / 2) * img_h
            x2 = (cx + w / 2) * img_w
            y2 = (cy + h / 2) * img_h

            boxes.append((cls, x1, y1, x2, y2))

    return boxes

def boxes_in_tile(boxes, tx1, ty1, tx2, ty2):
    tile_boxes = []
    tw = tx2 - tx1
    th = ty2 - ty1

    for cls, bx1, by1, bx2, by2 in boxes:
        orig_area = (bx2 - bx1) * (by2 - by1)
        if orig_area <= 0:
            continue

        ix1 = max(bx1, tx1)
        iy1 = max(by1, ty1)
        ix2 = min(bx2, tx2)
        iy2 = min(by2, ty2)

        if ix1 >= ix2 or iy1 >= iy2:
            continue

        clip_area = (ix2 - ix1) * (iy2 - iy1)
        if clip_area / orig_area < MIN_BBOX_AREA_RATIO:
            continue

        lx1 = ix1 - tx1
        ly1 = iy1 - ty1
        lx2 = ix2 - tx1
        ly2 = iy2 - ty1

        if (lx2 - lx1) < MIN_BBOX_PX or (ly2 - ly1) < MIN_BBOX_PX:
            continue

        cx = ((lx1 + lx2) / 2) / tw
        cy = ((ly1 + ly2) / 2) / th
        w = (lx2 - lx1) / tw
        h = (ly2 - ly1) / th

        tile_boxes.append((cls, cx, cy, w, h))

    return tile_boxes

tiled_dir = os.path.join(WORK_DIR, f"dataset_tiled_{TILE_SIZE}")

if os.path.exists(tiled_dir):
    shutil.rmtree(tiled_dir)

stride = int(TILE_SIZE * (1 - OVERLAP_RATIO))

for split in ["train", "val"]:
    src_img_dir = os.path.join(WORK_DIR, "images", split)
    src_label_dir = os.path.join(WORK_DIR, "labels", split)

    dst_img_dir = os.path.join(tiled_dir, "images", split)
    dst_label_dir = os.path.join(tiled_dir, "labels", split)

    os.makedirs(dst_img_dir, exist_ok=True)
    os.makedirs(dst_label_dir, exist_ok=True)

    img_files = sorted([f for f in os.listdir(src_img_dir) if f.endswith(IMG_EXTS)])

    positive_tiles = []
    negative_tiles = []

    for img_file in img_files:
        img_path = os.path.join(src_img_dir, img_file)
        img = cv2.imread(img_path)

        if img is None:
            continue

        img_h, img_w = img.shape[:2]
        stem = os.path.splitext(img_file)[0]
        label_path = os.path.join(src_label_dir, f"{stem}.txt")

        boxes = parse_yolo_label(label_path, img_w, img_h)

        y_starts = list(range(0, max(1, img_h - TILE_SIZE + 1), stride))
        x_starts = list(range(0, max(1, img_w - TILE_SIZE + 1), stride))

        if not y_starts:
            y_starts = [0]
        if not x_starts:
            x_starts = [0]

        if y_starts[-1] + TILE_SIZE < img_h:
            y_starts.append(img_h - TILE_SIZE)
        if x_starts[-1] + TILE_SIZE < img_w:
            x_starts.append(img_w - TILE_SIZE)

        for yi, y in enumerate(y_starts):
            for xi, x in enumerate(x_starts):
                tx1, ty1 = x, y
                tx2, ty2 = x + TILE_SIZE, y + TILE_SIZE

                tile_img = img[max(0, ty1):min(ty2, img_h), max(0, tx1):min(tx2, img_w)]

                if tile_img.shape[0] < TILE_SIZE or tile_img.shape[1] < TILE_SIZE:
                    padded = np.zeros((TILE_SIZE, TILE_SIZE, 3), dtype=np.uint8)
                    padded[:tile_img.shape[0], :tile_img.shape[1]] = tile_img
                    tile_img = padded

                tile_boxes = boxes_in_tile(boxes, tx1, ty1, tx2, ty2)

                tile_name = f"{stem}_t{yi:02d}{xi:02d}"
                tile_entry = (tile_img, tile_name, tile_boxes)

                if tile_boxes:
                    positive_tiles.append(tile_entry)
                else:
                    negative_tiles.append(tile_entry)

    n_pos = len(positive_tiles)
    n_neg_keep = max(1, int(n_pos * NEGATIVE_TILE_RATIO)) if negative_tiles and n_pos > 0 else 0
    n_neg_keep = min(n_neg_keep, len(negative_tiles))

    sampled_negatives = random.sample(negative_tiles, n_neg_keep) if n_neg_keep > 0 else []

    all_tiles = positive_tiles + sampled_negatives
    box_count = 0

    for tile_img, tile_name, tile_boxes in all_tiles:
        cv2.imwrite(os.path.join(dst_img_dir, f"{tile_name}.png"), tile_img)

        label_lines = [
            f"{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}"
            for cls, cx, cy, w, h in tile_boxes
        ]

        with open(os.path.join(dst_label_dir, f"{tile_name}.txt"), "w") as f:
            f.write("\n".join(label_lines))

        box_count += len(tile_boxes)

    print(
        f"{split:5s} | 원본 {len(img_files)}장 -> 타일 {len(all_tiles)}장 "
        f"(pos {n_pos}, neg {len(negative_tiles)}->{n_neg_keep}) | bbox {box_count}개"
    )

    if len(all_tiles) == 0:
        raise RuntimeError(f"{split} 타일 데이터가 0장입니다.")

data_yaml = {
    "path": tiled_dir,
    "train": "images/train",
    "val": "images/val",
    "nc": len(CLASS_NAMES),
    "names": CLASS_NAMES,
}

data_yaml_path = os.path.join(tiled_dir, "data.yaml")

with open(data_yaml_path, "w") as f:
    yaml.dump(data_yaml, f, allow_unicode=True, sort_keys=False)

print(f"\ndata.yaml: {data_yaml_path}")

train | 원본 8336장 -> 타일 40953장 (pos 39003, neg 61021->1950) | bbox 52328개
val   | 원본 927장 -> 타일 4649장 (pos 4428, neg 6696->221) | bbox 5918개

data.yaml: /content/checkmite_mvp1v11/dataset_tiled_640/data.yaml


In [ ]:
# === Cell 7: YOLOv8m 학습 ===
from ultralytics import YOLO

model = YOLO(MODEL)

results = model.train(
    data=data_yaml_path,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    device=0,
    project=RUN_PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    save_period=10,
    degrees=45,
    flipud=0.5,
    fliplr=0.5,
    mosaic=1.0,
    close_mosaic=10,
)


Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/checkmite_mvp1v11/dataset_tiled_640/data.yaml, degrees=45, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=mvp1v11, nbs=64, nms=False, opset=None, optimize=False, optimizer=au

In [ ]:
# === Cell 8: 학습 결과 시각화 ===
from IPython.display import Image, display
import pandas as pd
import os

run_path = os.path.join(RUN_PROJECT, RUN_NAME)

print("=== 학습 결과 ===")
results_png = os.path.join(run_path, 'results.png')
if os.path.exists(results_png):
    display(Image(results_png))
else:
    print(f"results.png 없음: {results_png}")

print("\n=== 혼동 행렬 ===")
cm_path = os.path.join(run_path, 'confusion_matrix.png')
if os.path.exists(cm_path):
    display(Image(cm_path))
else:
    print(f"confusion_matrix.png 없음: {cm_path}")

csv_path = os.path.join(run_path, 'results.csv')
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    if 'metrics/mAP50(B)' in df.columns:
        best_idx = df['metrics/mAP50(B)'].idxmax()
        best = df.iloc[best_idx]
        print(f"\n=== Best Epoch: {int(best['epoch'])} ===")
        print(f"  mAP50:    {best['metrics/mAP50(B)']:.4f}")
        print(f"  mAP50-95: {best['metrics/mAP50-95(B)']:.4f}")
        print(f"  Precision: {best['metrics/precision(B)']:.4f}")
        print(f"  Recall:    {best['metrics/recall(B)']:.4f}")
    else:
        print('results.csv에 metrics/mAP50(B) 컬럼이 없습니다.')
else:
    print(f"results.csv 없음: {csv_path}")


In [ ]:
# === Cell 9: 결과 저장 ===
import os, shutil, zipfile

os.makedirs(SAVE_DIR, exist_ok=True)

run_path = os.path.join(RUN_PROJECT, RUN_NAME)

# 주요 파일 복사
save_files = [
    'weights/best.pt',
    'weights/last.pt',
    'results.csv',
    'results.png',
    'confusion_matrix.png',
    'confusion_matrix_normalized.png',
    'args.yaml',
]

for fname in save_files:
    src = os.path.join(run_path, fname)
    if os.path.exists(src):
        dst = os.path.join(SAVE_DIR, os.path.basename(fname))
        shutil.copy2(src, dst)
        size_mb = os.path.getsize(dst) / 1024 / 1024
        print(f"  {os.path.basename(fname):40s} -> {size_mb:.1f} MB")
    else:
        print(f"  없음: {src}")

# data.yaml도 같이 저장
if os.path.exists(data_yaml_path):
    shutil.copy2(data_yaml_path, os.path.join(SAVE_DIR, 'data.yaml'))

# ZIP 패키징
zip_path = os.path.join(SAVE_DIR, f'{RUN_NAME}_output.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in save_files:
        fpath = os.path.join(SAVE_DIR, os.path.basename(fname))
        if os.path.exists(fpath):
            zf.write(fpath, os.path.basename(fname))
    data_yaml_saved = os.path.join(SAVE_DIR, 'data.yaml')
    if os.path.exists(data_yaml_saved):
        zf.write(data_yaml_saved, 'data.yaml')

print(f"\nZIP 저장: {zip_path}")
print(f"크기: {os.path.getsize(zip_path) / 1024 / 1024:.1f} MB")


In [ ]:
!zip -r /content/merged_dataset.zip /content/checkmite_mvp1v11/merged_dataset

스트리밍 출력 내용이 길어서 마지막 5000줄이 삭제되었습니다.
  adding: content/checkmite_mvp1v11/merged_dataset/labels/All/yolo_export_task30_20260505__Train__260324204901_frame_005.50s.txt (deflated 38%)
  adding: content/checkmite_mvp1v11/merged_dataset/labels/All/yolo_export_task37_20260521__Train__260405182713_frame_013.50s.txt (deflated 47%)
  adding: content/checkmite_mvp1v11/merged_dataset/labels/All/yolo_export_task30_20260505__Train__260324180001_frame_003.50s.txt (deflated 5%)
  adding: content/checkmite_mvp1v11/merged_dataset/labels/All/yolo_export_task30_20260505__Train__260324202818_frame_023.00s.txt (deflated 50%)
  adding: content/checkmite_mvp1v11/merged_dataset/labels/All/yolo_export_task37_20260521__Train__260331163452_frame_004.50s.txt (deflated 38%)
  adding: content/checkmite_mvp1v11/merged_dataset/labels/All/yolo_export_task38_20260505__Train__260406163002_frame_000.50s.txt (deflated 38%)
  adding: content/checkmite_mvp1v11/merged_dataset/labels/All/yolo_export_task37_20260521__Train__260

In [ ]:
!rm -rf /content/checkmite_mvp1v11/extracted_zips
!rm -rf /content/checkmite_mvp1v11/images
!rm -rf /content/checkmite_mvp1v11/labels